In [1]:
#!pip install -U datasets pandas sentence-transformers faiss-cpu pyarrow tqdm ipywidgets jsonlines
#!pip install "datasets<4.0.0"

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from datasets import load_dataset, load_from_disk
from sentence_transformers import SentenceTransformer
import faiss

In [8]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / ".git").exists() and (path / "README.md").exists():
            return path
    raise RuntimeError("Project root not found")


PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PREPARED_DIR = DATA_DIR / "prepared"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = PROJECT_DIR / "models" / "ru-news-mpnet-paraphrase-mnrl" / "final"

In [12]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = find_project_root()
lenta_path = PROJECT_DIR / "data" / "raw" / "lenta.csv"

df = pd.read_csv(
    lenta_path,
    dtype={
        "url": "string",
        "title": "string",
        "text": "string",
        "topic": "string",
        "tags": "string",
        "date": "string",
    },
    low_memory=False,
)

print(df.shape)
print(df.columns)
display(df.head())

(800975, 6)
Index(['url', 'title', 'text', 'topic', 'tags', 'date'], dtype='str')


,url,title,text,topic,tags,date
0,https://lenta.ru/news/1914/09/16/hungarnn/,1914. Русские войска вступили в пределы Венгрии,Бои у Сопоцкина и Друскеник закончились отступ...,Библиотека,Первая мировая,1914/09/16
1,https://lenta.ru/news/1914/09/16/lermontov/,1914. Празднование столетия М.Ю. Лермонтова от...,"Министерство народного просвещения, в виду про...",Библиотека,Первая мировая,1914/09/16
2,https://lenta.ru/news/1914/09/17/nesteroff/,1914. Das ist Nesteroff!,"Штабс-капитан П. Н. Нестеров на днях, увидев в...",Библиотека,Первая мировая,1914/09/17
3,https://lenta.ru/news/1914/09/17/bulldogn/,1914. Бульдог-гонец под Льежем,Фотограф-корреспондент Daily Mirror рассказыва...,Библиотека,Первая мировая,1914/09/17
4,https://lenta.ru/news/1914/09/18/zver/,1914. Под Люблином пойман швабский зверь,"Лица, приехавшие в Варшаву из Люблина, передаю...",Библиотека,Первая мировая,1914/09/18


In [13]:
df["published_at"] = pd.to_datetime(df["date"], format="%Y/%m/%d", errors="coerce")

df = df.dropna(subset=["title", "text", "published_at"]).copy()

df["title"] = df["title"].astype(str).str.strip()
df["text"] = df["text"].astype(str).str.strip()
df["topic"] = df["topic"].astype(str).str.strip()
df["tags"] = df["tags"].astype(str).str.strip()

df = df[
    (df["text"].str.len() > 300) &
    (df["published_at"] >= "2015-01-01")
].copy()

df["content"] = df["title"] + ". " + df["text"]

print(df.shape)
display(df.head())
print(df["topic"].value_counts().head(20))

(270100, 8)


,url,title,text,topic,tags,date,published_at,content
530629,https://lenta.ru/news/2015/01/01/russia/,Российские хоккеисты узнали соперника по 1/4 ф...,Молодежная сборная России по хоккею узнала соп...,Спорт,Летние виды,2015/01/01,2015-01-01,Российские хоккеисты узнали соперника по 1/4 ф...
530630,https://lenta.ru/news/2015/01/01/four/,Задержаны подозреваемые в убийстве четырех чел...,Сотрудники полиции задержали двух жителей Сара...,Силовые структуры,Криминал,2015/01/01,2015-01-01,Задержаны подозреваемые в убийстве четырех чел...
530631,https://lenta.ru/news/2015/01/01/plane/,Россия направила в Индонезию спасателей для по...,МЧС России направило в Индонезию два самолета ...,Россия,Происшествия,2015/01/01,2015-01-01,Россия направила в Индонезию спасателей для по...
530632,https://lenta.ru/news/2015/01/01/migrantpatent/,Вступил в силу закон для мигрантов о получении...,В России с 1 января 2015 года вступили в силу ...,Россия,Политика,2015/01/01,2015-01-01,Вступил в силу закон для мигрантов о получении...
530633,https://lenta.ru/news/2015/01/01/oficer/,Суд арестовал подозреваемого в шпионаже литовс...,Участковый суд Вильнюса арестовал подозреваемо...,Бывший СССР,Прибалтика,2015/01/01,2015-01-01,Суд арестовал подозреваемого в шпионаже литовс...


topic
Россия               32407
Мир                  30366
Спорт                20659
Силовые структуры    16487
Бывший СССР          16095
Экономика            14767
Культура             13172
Интернет и СМИ       12928
Наука и техника      12221
Из жизни              8622
Ценности              7762
Бизнес                7253
Дом                   6729
Путешествия           6404
69-я параллель        1268
Крым                   666
Культпросвет           340
Легпром                113
Библиотека               6
Оружие                   3
Name: count, dtype: int64


In [14]:
sample_df = (
    df.sort_values("published_at")
      .sample(n=min(5000, len(df)), random_state=42)
      .reset_index(drop=True)
)

sample_df[["title", "topic", "tags", "published_at"]].head()

,title,topic,tags,published_at
0,Камень в кольце с барахолки оказался стоящим с...,Из жизни,Люди,2017-05-22
1,Автор люстры из гигиенических тампонов создала...,Ценности,Часы,2015-11-16
2,Разрушение крыла Су-57 попало на видео,Наука и техника,Оружие,2018-11-07
3,Вратарь нырнул за мячом «рыбкой» и получил гол...,NaN,Футбол,2019-03-31
4,Два взрыва прогремели на дороге к «Саур-Могиле...,Бывший СССР,Украина,2017-05-08


In [15]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(str(MODEL_PATH))

texts = sample_df["content"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

print(embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

(5000, 768)


In [16]:
import faiss

dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)  # cosine (т.к. normalized)
index.add(embeddings)

print("Vectors in index:", index.ntotal)

Vectors in index: 5000


In [26]:
def search_similar(article_idx: int, top_k: int = 6):
    query_vec = embeddings[article_idx:article_idx + 1]
    scores, indices = index.search(query_vec, top_k)

    result = sample_df.iloc[indices[0]].copy()
    result["score"] = scores[0]
    return result[["title", "topic", "published_at", "score"]]

def search_by_query(query: str, top_k=5):
    query_emb = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_emb, top_k)

    result = sample_df.iloc[indices[0]].copy()
    result["score"] = scores[0]
    return result[["title", "topic", "score", "published_at"]]

In [19]:
article_idx = 111

print("QUERY:")
print(sample_df.loc[article_idx, "title"])
print()

search_similar(article_idx, top_k=6)

QUERY:
Британские модели устроили съемку рядом с телом самоубийцы



,title,topic,published_at,score
111,Британские модели устроили съемку рядом с тело...,Из жизни,2016-12-23,1.000000
2042,Британка сделала макияж спящему возлюбленному,Из жизни,2017-05-18,0.503342
4589,Блогеры пересняли трейлер «Отряда самоубийц» в...,Интернет и СМИ,2016-09-08,0.489689
4379,Режиссер «Отряда самоубийц» рассказал о парали...,Культура,2016-03-18,0.480572
4812,BFM TV заснял процесс изготовления статуэток «...,Культура,2017-02-20,0.466185
1368,Модель plus-size предстала в образе развратной...,Ценности,2018-01-19,0.463882


In [20]:
i = 100
j = 101

print(sample_df.loc[i, "title"])
print(sample_df.loc[j, "title"])

from sentence_transformers import util

emb = model.encode(
    [sample_df.loc[i, "content"], sample_df.loc[j, "content"]],
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(util.cos_sim(emb[0], emb[1]))

Лукашенко сделал Путину предложение по интеграции
Глава ЦРУ усомнился в возможностях США понять Ближний Восток
tensor([[0.2603]], device='cuda:0')


In [29]:
search_by_query("война на ближнем востоке")

,title,topic,score,published_at
4087,Иран счел планы Трампа усилить войска на Ближн...,NaN,0.497189,2019-09-22
4982,Вашингтон исключил конфликт с Россией в Сирии,Мир,0.453526,2015-10-13
101,Глава ЦРУ усомнился в возможностях США понять ...,Мир,0.451244,2016-07-14
4179,В США испугались «абсолютного оружия» России в...,NaN,0.449656,2019-11-24
3132,Генерал назвал итог возможной войны между Росс...,Наука и техника,0.448670,2018-04-06


In [30]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Сортируем как поток
stream_df = sample_df.sort_values("published_at").reset_index(drop=True)
stream_embeddings = embeddings[stream_df.index].astype("float32")

In [31]:
texts = stream_df["content"].tolist()

stream_embeddings = model.encode(
    texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [40]:
import faiss

SIM_THRESHOLD = 0.75  # можно потом крутить: 0.65 / 0.70 / 0.75
MAX_DAYS_DIFF = 5  # или 3 / 14
TOP_K = 5

article_topic = []
topics = []
topic_articles = {}

dim = stream_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

for i in tqdm(range(len(stream_df))):
    emb = stream_embeddings[i:i+1]

    if index.ntotal == 0:
        topic_id = 0
        topics.append({
            "topic_id": topic_id,
            "representative_article_idx": i,
            "title": stream_df.loc[i, "title"],
            "created_at": stream_df.loc[i, "published_at"],
            "last_article_at": stream_df.loc[i, "published_at"],
            "article_count": 1,
        })
        topic_articles[topic_id] = [i]
        article_topic.append(topic_id)
        index.add(emb)
        continue

    scores, indices = index.search(emb, min(TOP_K, index.ntotal))
    time_diff = abs((stream_df.loc[i, "published_at"] - stream_df.loc[best_neighbor_idx, "published_at"]).days)

    best_score = float(scores[0][0])
    best_neighbor_idx = int(indices[0][0])

    if best_score >= SIM_THRESHOLD and time_diff <= MAX_DAYS_DIFF:
        topic_id = article_topic[best_neighbor_idx]
        topic_articles[topic_id].append(i)
        article_topic.append(topic_id)

        topics[topic_id]["article_count"] += 1
        topics[topic_id]["last_article_at"] = stream_df.loc[i, "published_at"]
    else:
        topic_id = len(topics)
        topics.append({
            "topic_id": topic_id,
            "representative_article_idx": i,
            "title": stream_df.loc[i, "title"],
            "created_at": stream_df.loc[i, "published_at"],
            "last_article_at": stream_df.loc[i, "published_at"],
            "article_count": 1,
        })
        topic_articles[topic_id] = [i]
        article_topic.append(topic_id)

    index.add(emb)

stream_df["topic_id_auto"] = article_topic
topics_df = pd.DataFrame(topics)

print("Articles:", len(stream_df))
print("Topics:", len(topics_df))
print("Avg articles per topic:", len(stream_df) / len(topics_df))
display(topics_df.sort_values("article_count", ascending=False).head(20))

100%|██████████| 5000/5000 [00:01<00:00, 2799.08it/s]

Articles: 5000
Topics: 4995
Avg articles per topic: 1.001001001001001


,topic_id,representative_article_idx,title,created_at,last_article_at,article_count
2055,2055,2056,Apple подала в суд на своего поставщика чипов,2017-01-21,2019-04-17,2
578,578,579,Объем предложения апартаментов в «старой» Моск...,2015-10-07,2019-08-22,2
1035,1035,1036,ЦБ пригрозил отзывом лицензии страховщикам без...,2016-03-23,2017-05-24,2
263,263,263,Сумма исков «Газпрома» к «Нафтогазу» составила...,2015-05-05,2015-05-28,2
721,721,722,Курс евро впервые за последние семь дней превы...,2015-11-23,2017-06-07,2
4963,4963,4968,Более 400 елочных базаров откроются в Подмосковье,2019-12-06,2019-12-06,1
18,18,18,Двух мигрантов в Испании заподозрили в убийств...,2015-01-15,2015-01-15,1
4979,4979,4984,Названа причина смерти двухлетней россиянки во...,2019-12-11,2019-12-11,1
1,1,1,Иран опроверг информацию о соглашении с США об...,2015-01-03,2015-01-03,1
4,4,4,Пушков предложил принять в Европе закон по защ...,2015-01-11,2015-01-11,1


In [39]:
def show_topic(topic_id, max_rows=20):
    idxs = topic_articles[topic_id]
    result = stream_df.iloc[idxs].copy()
    return result[["published_at", "title", "topic", "tags", "topic_id_auto"]].head(max_rows)

top_topic_ids = topics_df.sort_values("article_count", ascending=False)["topic_id"].head(10).tolist()

for tid in top_topic_ids:
    print("=" * 100)
    print("TOPIC", tid, "|", topics_df.loc[tid, "title"], "| count:", topics_df.loc[tid, "article_count"])
    display(show_topic(tid, max_rows=10))

TOPIC 1680 | Пилот Mercedes Росберг выиграл квалификацию Гран-при Японии | count: 2


,published_at,title,topic,tags,topic_id_auto
1685,2016-10-08,Пилот Mercedes Росберг выиграл квалификацию Гр...,Спорт,Летние виды,1680
1796,2016-11-13,Хэмилтон впервые в карьере выиграл Гран-при Бр...,Спорт,Летние виды,1680


TOPIC 514 | На востоке Турции атакован армейский конвой | count: 2


,published_at,title,topic,tags,topic_id_auto
516,2015-09-15,На востоке Турции атакован армейский конвой,Мир,Политика,514
1572,2016-08-27,Курды обстреляли турецкую танковую колонну в С...,Мир,Конфликты,514


TOPIC 262 | Сумма исков «Газпрома» к «Нафтогазу» составила 23,8 миллиарда долларов | count: 2


...

,published_at,title,topic,tags,topic_id_auto
2056,2017-01-21,Apple подала в суд на своего поставщика чипов,Бизнес,Бизнес,2049
4155,2019-04-17,Apple решила откупиться от главного противника,NaN,Бизнес,2049


TOPIC 212 | Прокатчик «Номера 44» согласился с решением Минкульта о запрете фильма | count: 2


,published_at,title,topic,tags,topic_id_auto
212,2015-04-15,Прокатчик «Номера 44» согласился с решением Ми...,Культура,Кино,212
219,2015-04-16,Киргизия вслед за Россией и Белоруссией отказа...,Культура,Кино,212


TOPIC 1135 | Актер из «Фантастических зверей» намекнул на появление молодого Дамблдора | count: 2


,published_at,title,topic,tags,topic_id_auto
1138,2016-04-20,Актер из «Фантастических зверей» намекнул на п...,Культура,Кино,1135
4581,2019-08-20,Актеров из «Гарри Поттера» заподозрили в романе,NaN,Кино,1135


In [41]:
def assign_roles_for_topic(topic_id):
    idxs = topic_articles[topic_id]
    if len(idxs) == 1:
        return {idxs[0]: ("new_topic", 1.0, None)}

    roles = {}
    seen = []

    for idx in idxs:
        if not seen:
            roles[idx] = ("new_topic", 1.0, None)
            seen.append(idx)
            continue

        current_emb = stream_embeddings[idx:idx+1]
        seen_embs = stream_embeddings[seen]

        sims = (current_emb @ seen_embs.T)[0]
        nearest_pos = int(np.argmax(sims))
        nearest_sim = float(sims[nearest_pos])
        nearest_idx = seen[nearest_pos]

        novelty_score = 1.0 - nearest_sim

        if nearest_sim >= 0.90:
            role = "duplicate"
        elif nearest_sim >= 0.78:
            role = "minor_update"
        else:
            role = "major_update"

        roles[idx] = (role, novelty_score, nearest_idx)
        seen.append(idx)

    return roles


all_roles = {}

for tid in topic_articles:
    all_roles.update(assign_roles_for_topic(tid))

stream_df["role"] = [all_roles[i][0] for i in range(len(stream_df))]
stream_df["novelty_score"] = [all_roles[i][1] for i in range(len(stream_df))]
stream_df["nearest_previous_idx"] = [all_roles[i][2] for i in range(len(stream_df))]

In [35]:
updates = stream_df[
    stream_df["role"].isin(["minor_update", "major_update"])
].sort_values("novelty_score", ascending=False)

display(updates[["published_at", "title", "topic", "tags", "topic_id_auto", "role", "novelty_score"]].head(30))

,published_at,title,topic,tags,topic_id_auto,role,novelty_score
3324,2018-04-12,Потребительский оптимизм заставил россиян акти...,Дом,Квартира,2260,major_update,0.279726
4336,2019-06-14,Йеменские повстанцы атаковали гражданский аэро...,NaN,Конфликты,1529,major_update,0.279622
1507,2016-08-09,«Газпром» увеличил транзит через Украину из-за...,Бизнес,Деловой климат,41,major_update,0.279595
2031,2017-01-14,Подавляющее большинство американцев посчитали ...,Мир,Общество,886,major_update,0.278708
4259,2019-05-20,Речь Зеленского расстроила команду Порошенко,NaN,Украина,3945,major_update,0.278038
1591,2016-09-05,Москва оказалась на третьем месте в Европе по ...,Дом,Офис,71,major_update,0.278034
1665,2016-10-02,Хоуситы подбили ракетой предоставленный америк...,Силовые структуры,NaN,1529,major_update,0.277986
4250,2019-05-18,Движение поездов по Крымскому мосту откроют до...,NaN,Общество,2026,major_update,0.277783
2020,2017-01-12,Потенциальный министр обороны США назвал Росси...,Мир,Политика,1663,major_update,0.277681
4475,2019-07-22,Турция рассказала о планах на С-400,NaN,Политика,3194,major_update,0.277659


In [36]:
period_df = df[
    (df["published_at"] >= "2018-06-01") &
    (df["published_at"] < "2018-07-01")
].copy()

period_df = period_df.reset_index(drop=True)

print(period_df.shape)
display(period_df[["published_at", "title", "topic", "tags"]].head())

(3429, 8)


,published_at,title,topic,tags
0,2018-06-01,Донецкие «Чебурашки» отомстили за гибель коман...,Бывший СССР,Украина
1,2018-06-01,Россию завалило первым летним снегом,Россия,Общество
2,2018-06-01,Павла Дурова обвинили в краже чужой разработки,Интернет и СМИ,Интернет
3,2018-06-01,Россиян оставят без дешевых квартир,Экономика,Госэкономика
4,2018-06-01,Хирург тайно вырезал британке орган и довел ее...,Из жизни,Люди
